<a href="https://colab.research.google.com/github/OlhaZahrebelna/python_for_ds_task/blob/main/HW_Zahrebelna_Olha%22%D0%92%D0%B8%D0%BA%D0%BE%D1%80%D0%B8%D1%81%D1%82%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BF%D1%80%D0%BE%D0%BC%D0%BF%D1%82%D1%96%D0%B2_%D1%96_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%96%D0%B2_%D0%B2_Langchain_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [1]:
!pip -q install \
langchain \
langchain-openai \
langchain-community \
langchain-experimental \
openai \
python-dotenv \
ddgs

In [2]:
import os
import json
import re


from langchain_openai import OpenAI, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.agents import create_agent
from langchain_experimental.utilities import PythonREPL
from langchain_classic import hub
from langchain_classic.agents import load_tools, Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain_classic.chains import LLMMathChain
from langchain_experimental.utilities import PythonREPL

In [3]:
with open('creds.json') as file:
  creds = json.load(file)

os.environ["OPENAI_API_KEY"] = creds["OPENAI_API_KEY"]

In [4]:
overal_temperature = 0.3
llm = OpenAI(temperature=overal_temperature,
             max_tokens=200)

In [5]:
print(llm.invoke("Explain the topic Quantum Computing. Include: a definition, key advantages, and current research. The response must be very brief (up to 200 characters)."))



Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. It has the potential to solve complex problems much faster than classical computers. Current research focuses on developing more stable and scalable quantum systems.


Обрав temperature=0.3, щоб отримати більш точну і стабільну відповідь без зайвої креативності.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [6]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template=(
        "Explain the topic '{topic}'. "
        "Include: definition, key advantages, and current research. "
        "Respond briefly and to the point."))

chain = prompt | llm

topic = ["Bayesian methods in machine learning",
    "Transformers in machine learning",
    "Explainable AI"]

In [7]:
for t in topic:
    print(f"Topic: {t}")
    response = chain.invoke({"topic": t})
    print(response)

Topic: Bayesian methods in machine learning


Bayesian methods in machine learning refer to a statistical approach that uses Bayes' theorem to update the probability of a hypothesis based on new evidence. It is a popular technique used in various machine learning algorithms, such as Bayesian networks, Gaussian processes, and Bayesian linear regression.

One of the key advantages of Bayesian methods is their ability to incorporate prior knowledge or beliefs into the learning process. This allows for more accurate and robust predictions, especially in cases where there is limited data available. Additionally, Bayesian methods can handle uncertainty and can provide a measure of confidence in the predictions.

Current research in Bayesian methods in machine learning is focused on developing more efficient algorithms and techniques for handling large datasets. There is also a growing interest in applying Bayesian methods to deep learning, as it can help address some of the limitations of tr



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [8]:
ddg = DuckDuckGoSearchResults(output_format="list")

@tool
def search_scientific_papers(topic: str) -> str:
    """Search for recent scientific publications on a given topic."""

    query = f'site:arxiv.org OR site:openreview.net OR site:doi.org "{topic}" scientific paper'
    results = ddg.invoke(query)

    if not results:
        return f"No results found for topic: {topic}"

    formatted_results = []
    for i, item in enumerate(results[:5], 1):
        title = item.get("title", "No title")
        link = item.get("link", "No link")
        snippet = item.get("snippet", "No description available")

        formatted_results.append(
            f"{i}. Title: {title}\n"
            f"Link: {link}\n"
            f"Description: {snippet}"
        )

    return "\n\n".join(formatted_results)

In [9]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [10]:
agent = create_agent(
    model=llm,
    tools=[search_scientific_papers],
    system_prompt=(
        "You are a scientific research assistant. "
        "Use the tool to find 5 recent scientific publications. "
        "Return the answer in English. "
        "For each publication provide: title, authors if available, and a short description. "
        "Do not invent facts."
    )
)

In [11]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find 5 recent scientific publications on the topic of artificial intelligence."# that need to search
            }
        ]
    }
)

print(response["messages"][-1].content)

Here are 5 recent scientific publications on the topic of artificial intelligence:

1. **Title:** AI4Research: A Survey of Artificial Intelligence for Scientific Research  
   **Authors:** Qiguang Chen et al.  
   **Description:** This paper surveys recent advancements in artificial intelligence (AI), particularly focusing on large language models (LLMs) and their applications in scientific research.  
   **Link:** [Read more](https://arxiv.org/abs/2507.01903)

2. **Title:** The AI Scientist-v2: Workshop-Level Automated Scientific Discovery via ...  
   **Authors:** Not specified  
   **Description:** This publication discusses the role of AI in scientific discovery, including aspects of AI safety and its implications for various fields.  
   **Link:** [Read more](https://arxiv.org/abs/2504.08066)

3. **Title:** Rethinking Science in the Age of Artificial Intelligence  
   **Authors:** Not specified  
   **Description:** This commentary explores how AI is reshaping research methodologi



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [12]:
python_repl = PythonREPL()
python_tool = Tool(
    name='python_repl',
    description=(
      "Execute Python code for business analytics tasks such as sales forecasting, "
      "time series analysis, data processing, and incorporating external factors "
      "like inflation and weather into predictions."
),
    func=python_repl.run
)

In [13]:
@tool
def search_inflation_data(country: str) -> str:
    """Search for current inflation rate and forecast for a given country."""

    query = f"{country} inflation rate 2025 forecast CPI"
    results = ddg.invoke(query)

    numbers = []
    for item in results:
        text = item.get("snippet", "")
        found = re.findall(r"\d+\.?\d*%", text)
        numbers.extend(found)

    if numbers:
        return f"Found inflation values: {', '.join(numbers)}"
    else:
        return "No precise inflation data found"

In [14]:
@tool
def search_weather_data(location: str) -> str:
    """Search for weather conditions, climate trends, or forecasts for a location."""

    query = f"{location} weather trends average temperature rainfall forecast"
    results = ddg.invoke(query)

    if not results:
        return f"No weather data found for {location}"

    output = []
    for i, item in enumerate(results[:5], 1):
        output.append(
            f"{i}. {item.get('title', '')}\n"
            f"{item.get('snippet', '')}\n"
            f"{item.get('link', '')}"
        )

    return "\n\n".join(output)

In [15]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[
        python_tool,
        search_inflation_data,
        search_weather_data
    ],
    system_prompt=(
        "You are a business analytics assistant. "
        "Use tools to retrieve external data such as inflation and weather. "
        "Only use values explicitly found in tool outputs. "
        "Do not invent percentages, coefficients, or assumptions. "
        "If an external factor cannot be quantified precisely, say that clearly. "
        "When forecasting, distinguish between data-based facts and rough assumptions. "
        "Use the python tool for calculations."
        "Always include the exact value and source used from the tool output."
        )
)

In [16]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "We export oranges from Brazil. In 2021 we exported 200 tons, in 2022 - 190 tons, in 2023 - 210 tons, and in 2024 (which is not finished yet) - 220 tons. Estimate how many oranges we will be able to export in 2025, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation."
            }
        ]
    }
)

print(response["messages"][-1].content)

To estimate the export of oranges from Brazil in 2025, we can analyze the historical export data and consider the external factors such as weather conditions and inflation.

### Historical Export Data:
- 2021: 200 tons
- 2022: 190 tons
- 2023: 210 tons
- 2024: 220 tons

### Trend Analysis:
From the data, we can see a general upward trend in exports from 2023 to 2024. The exports increased from 210 tons in 2023 to 220 tons in 2024, which is a growth of 10 tons.

### Weather Conditions:
The weather in Brazil is generally favorable for orange cultivation, with moderate rainfall and temperatures that do not fall below freezing in winter. This suggests that weather conditions are likely to remain conducive for orange production.

### Inflation Data:
The inflation rates found for Brazil are as follows:
- 2021: 5.53%
- 2022: 0.26%
- 2023: 0.43%
- 2024: 0.23%

While inflation can impact costs and pricing, the specific effect on orange exports is not quantifiable without additional data on how 